In [ ]:
import pathlib
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from dataset_configs.smpl.utils import compute_B_from_beta, smpl_pose_to_D_orientation
from dataset_configs.realworld import consts as rw_consts
from dataset_configs.realworld.utils import (
    generate_df_dict,
    generate_default_placement_params,
    generate_B_range,
)
from wimusim import WIMUSim, Optimizer, utils

## Introduction
The parameter identification process in WIMUSim involves optimizing WIMUSim's parameters
to closely replicate real-world IMU data. This process ensures that the simulated IMU data
reflects real-world sensor dynamics as accurately as possible. The goal is to minimize the
discrepancy between real and simulated data by fine-tuning the four primary parameters:
**Body (B)**, **Dynamics (D)**, **Placement (P)**, and **Hardware (H)**.

### What We Will Do in This Notebook
In this notebook, we demonstrate the parameter identification process using the
**RealWorld** dataset with **SMPL** pose estimation from **HMR2.0 / 4D-Humans**:

1. **Load the Data**: Load synchronized real IMU data (RealWorld) and SMPL pose
   estimation output (HMR2.0) for the same subject/activity.
2. **Initialize Parameters**:
   - **B**: Bone vectors from SMPL β via `compute_B_from_beta`.
   - **D**: Joint orientations from SMPL pose via `smpl_pose_to_D_orientation`.
   - **P**: Default IMU placements from `generate_default_placement_params`.
   - **H**: Default hardware noise models.
3. **Define Optimization Configurations**: Set up loss functions and constraints.
4. **Run the Optimization**: Gradient-based optimization to match simulated and real IMU data.
5. **Analyze the Results**: Visualize the optimization progress and final parameter values.

## 1. Load Data

### RealWorld IMU Data
The **RealWorld** dataset contains synchronized IMU recordings (50 Hz) from 7 body
placements across 8 activities for 15 subjects.

Set `REALWORLD_PATH` to the root directory of the dataset.  Each subject's data lives
under `{REALWORLD_PATH}/{subject_id}/data/` as one ZIP file per placement.

### SMPL Pose Estimation
Run **HMR2.0** or **4D-Humans** on the RealWorld video for the same subject/activity
and save the output as a `.npz` file with keys:
- `betas`: shape parameters `(10,)`
- `global_orient`: root rotation matrices `(T, 3, 3)`
- `body_pose`: per-joint rotation matrices `(T, 23, 3, 3)`

In [ ]:
REALWORLD_PATH = "path/to/realworld"  # root directory of the RealWorld dataset
subject_id = "p001"
activity   = "walking"
recording_no = 1

# Load real IMU data for each placement
acc_dict  = {}
gyro_dict = {}
for imu_name, placement in rw_consts.IMU_NAME_PAIRS_DICT.items():
    zip_path = pathlib.Path(REALWORLD_PATH) / subject_id / "data" / f"acc_{activity}_{placement}.zip"
    df_dict  = generate_df_dict(str(zip_path), verbose=False, recording_no=recording_no)
    for df in df_dict.values():
        acc_cols  = [c for c in df.columns if "Acc"  in c]
        gyro_cols = [c for c in df.columns if "Gyro" in c]
        if acc_cols:
            acc_dict[imu_name]  = df[acc_cols].values.astype(np.float32)
        if gyro_cols:
            gyro_dict[imu_name] = df[gyro_cols].values.astype(np.float32)

n_samples = min(v.shape[0] for v in acc_dict.values())
print(f"Loaded {n_samples} samples at {rw_consts.IMU_FRAME_RATE} Hz for subject {subject_id}, activity '{activity}'")

# Load SMPL pose estimation output (HMR2.0 / 4D-Humans)
smpl_path = f"data/{subject_id}_{activity}_smpl.npz"  # adjust to your file path
smpl_data     = np.load(smpl_path)
betas         = smpl_data["betas"]          # (10,)
global_orient = smpl_data["global_orient"]  # (T, 3, 3)
body_pose     = smpl_data["body_pose"]      # (T, 23, 3, 3)
print(f"SMPL output loaded: betas {betas.shape}, global_orient {global_orient.shape}, body_pose {body_pose.shape}")

## 2. Initialize WIMUSim Parameters

### Step 2.1: Initialize Dynamics (D)
We initialize the **Dynamics (D)** parameters directly from the SMPL pose estimation
output using `smpl_pose_to_D_orientation`. This converts the SMPL rotation matrices to
per-joint quaternions in WXYZ format (WIMUSim convention).

> The video-based SMPL pose runs at a different frame rate (e.g. 30 Hz from HMR2.0)
> than the IMU (50 Hz). The optimization will handle the mismatch; alternatively
> you can resample to match before passing to WIMUSim.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU.")

orientation = smpl_pose_to_D_orientation(global_orient, body_pose)
D_dict = {"orientation": orientation}

D = WIMUSim.Dynamics(
    **D_dict,
    device=device,
    sample_rate=30,   # HMR2.0 / 4D-Humans output rate
    requires_grad=True,
)

### Step 2.2: Initialize Body (B), Placement (P), and Hardware (H)

- **B**: Bone vectors computed from SMPL β using a T-pose forward pass.
- **P**: Default IMU placement offsets derived from the bone vectors.
- **H**: Default hardware noise/bias models.

In [ ]:
smpl_model_path = "path/to/smpl/models"  # directory containing SMPL_NEUTRAL.pkl

B_rp       = compute_B_from_beta(betas, smpl_model_path=smpl_model_path, gender="neutral")
B_rp_range = generate_B_range(B_rp)

B = WIMUSim.Body(
    rp=B_rp,
    rp_range_dict=B_rp_range,
    device=device,
    requires_grad=True,
)

P_params = generate_default_placement_params(B_rp)
P = WIMUSim.Placement(
    **P_params,
    device=device,
    requires_grad=True,
)

H = WIMUSim.Hardware(
    **utils.generate_default_H_configs(P.imu_names),
    device=device,
    requires_grad=True,
)

### Step 3: **Launch the WIMUSim Environment**
Finally, we create an instance of the WIMUSim environment using the initialized B, D, P, and H parameters. With this setup, the WIMUSim environment is now you can generate virtual IMU data by running the wimusim_env.simulate() method.

In [ ]:
wimusim_env = WIMUSim(B=B, D=D, P=P, H=H, dataset_name="SMPL")

In [ ]:
# Visualize pre-optimization virtual IMU for LLA (left lower arm)
virtual_IMU_dict = wimusim_env.simulate(mode="generate")

fig, axs = plt.subplots(3, 2, figsize=(15, 6))
acc_LLA_pre_opt, gyro_LLA_pre_opt = virtual_IMU_dict["LLA"]
start, end = 0, 500  # 10 sec at 50 Hz
axs[0, 0].set_title("Virtual Accelerometer (m/s²)")
axs[0, 1].set_title("Virtual Gyroscope (rad/s)")
for i in range(3):
    axs[i, 0].plot(acc_LLA_pre_opt.detach().cpu().numpy()[start:end, i], label=f"Acc{['X', 'Y', 'Z'][i]}")
    axs[i, 0].legend()
    axs[i, 1].plot(gyro_LLA_pre_opt.detach().cpu().numpy()[start:end, i], label=f"Gyro{['X', 'Y', 'Z'][i]}")
    axs[i, 1].legend()
plt.tight_layout()
plt.show()

## 4. Initialize the Optimizer

Next, we set up the optimizer, which is responsible for updating the WIMUSim parameters to minimize the difference between the real and virtual IMU data.


In [9]:
opt = Optimizer(
        wimusim_env,
        # These meta info is optional (only for logging purpose)
        meta_info={
            "subject_id": subject_id,
            "scenario_type": scenario_type,
        },
    )

### 4.1 Set the target IMU data
To perform the parameter identification, we need to set the **target IMU data**, which is the real IMU data we want to match. This includes acceleration and gyroscope readings for each IMU in the REALDISP dataset.

In [ ]:
target_IMU_dict = {
    imu_name: (
        torch.tensor(acc_dict[imu_name][:n_samples],  device=wimusim_env.device),
        torch.tensor(gyro_dict[imu_name][:n_samples], device=wimusim_env.device),
    )
    for imu_name in wimusim_env.P.imu_names
    if imu_name in acc_dict and imu_name in gyro_dict
}

opt.set_target_IMU_dict(target_IMU_dict)

### Step 4.2: **Define other optimization configurations**

To fine-tune the parameters, we need to define some optimization configurations, including learning rates, loss weights, and scheduling strategies.

In [ ]:
opt.init_optimizers()
opt.scheduler_dict = {
    "Do": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["Do"], T_0=2, T_mult=2, eta_min=1e-8, verbose=False
    ),
    "B": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["B"], T_0=2, T_mult=2, eta_min=1e-6, verbose=False
    ),
    "Pp": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["Pp"], T_0=2, T_mult=2, eta_min=1e-6, verbose=False
    ),
    "Po": torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt.optimizer_dict["Pp"], T_0=2, T_mult=2, eta_min=1e-6, verbose=False
    ),
}

# RealWorld IMU names: LHD (head), WST (waist), LCH (chest), LUA (upper arm),
#                      LLA (forearm), LTH (thigh), LSH (shin)
opt.rmse_weight_dict = {
    "LLA": (2.0, 3.0),
    "LUA": (1.0, 2.0),
    "LHD": (1.0, 1.0),
    "LCH": (1.0, 1.0),
    "WST": (1.0, 1.0),
    "LTH": (1.0, 2.0),
    "LSH": (2.0, 3.0),
}

opt.loss_coeff_dict["sym"] = 1e1
opt.loss_coeff_dict["rom"] = 1e1
opt.loss_coeff_dict["b_range"] = 1e2
opt.loss_coeff_dict["p_range"] = 1e3

opt.loss_coeff_dict  # show all loss coefficients

### Step 5: **Run the Optimization**

Finally, we run the optimization process. During this process, the optimizer will iteratively update the parameters to minimize the error between the real and virtual IMU data. The following code use wandb to log the optimization process, but you can disable it by setting `log_wandb=False`.

In [ ]:
wandb_project_name = "wimusim_realworld_param_opt"
rand_int = np.random.randint(1000)
wandb_run_name = f"{subject_id}_{activity}_{rand_int}"

_ = opt.fit(
    epochs=1024,
    early_stopping=False,
    log_wandb=True,
    wandb_project_config={
        "project_name": wandb_project_name,
        "run_name": wandb_run_name,
    }
)

## Visualization and Analysis

### 1. **Comparing Real and Virtual IMU Data**
To evaluate the effectiveness of the parameter identification process, we will visualize the real IMU data (`target_IMU`), the initial (pre-optimization) virtual IMU data, and the optimized virtual IMU data on the same plot. This comparison allows us to see how closely the virtual IMU data has aligned with the real IMU data after optimization.

The following plot compares the acc and gyro of target IMU (real data), pre-optimization virtual IMU, optimized virtual IMU for RLA IMU:

In [13]:
virtual_IMU_dict_opt = opt.env.simulate(mode="parameterise")

In [ ]:
def compare_real_virtual_IMU(target_IMU_dict, virtual_IMU_dict, imu_name="LLA", start=0, end=500):
    fig, axs = plt.subplots(3, 2, figsize=(15, 6))
    acc_real, gyro_real = target_IMU_dict[imu_name]
    acc_sim,  gyro_sim  = virtual_IMU_dict[imu_name]

    axs[0, 0].set_title(f"Real vs Virtual Accelerometer — {imu_name} (m/s²)")
    axs[0, 1].set_title(f"Real vs Virtual Gyroscope — {imu_name} (rad/s)")
    for i in range(3):
        label = ['X', 'Y', 'Z'][i]
        axs[i, 0].plot(acc_real.detach().cpu().numpy()[start:end, i], label=f"real_{label}")
        axs[i, 0].plot(acc_sim.detach().cpu().numpy()[start:end, i],  label=f"sim_{label}")
        axs[i, 0].legend()
        axs[i, 1].plot(gyro_real.detach().cpu().numpy()[start:end, i], label=f"real_{label}")
        axs[i, 1].plot(gyro_sim.detach().cpu().numpy()[start:end, i],  label=f"sim_{label}")
        axs[i, 1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare the real and virtual IMU (pre-opt) data for LLA
compare_real_virtual_IMU(target_IMU_dict, virtual_IMU_dict, imu_name="LLA")

In [ ]:
# Compare the real and virtual IMU (post-opt) data for LLA
compare_real_virtual_IMU(target_IMU_dict, virtual_IMU_dict_opt, imu_name="LLA")

### 2. **3D Visualization in PyBullet**
You can also visualize the moving humanoid in the 3D PyBullet environment. This visualization will show the human body model and the IMU placements, making it easy to see the overall movement patterns.

In [17]:
wimusim_env.launch_pybullet_client()
wimusim_env.run_visualization()

startThreads creating 1 threads.
starting thread 0
started thread 0 
argc=2
argv[0] = --unused
argv[1] = --start_demo_name=Physics Server
ExampleBrowserThreadFunc started
X11 functions dynamically loaded using dlopen/dlsym OK!
X11 functions dynamically loaded using dlopen/dlsym OK!
Creating context
Created GL 3.3 context
Direct GLX rendering context obtained
Making context current
GL_VENDOR=NVIDIA Corporation
GL_RENDERER=NVIDIA GeForce GTX 1080 Ti/PCIe/SSE2
GL_VERSION=3.3.0 NVIDIA 515.105.01
GL_SHADING_LANGUAGE_VERSION=3.30 NVIDIA via Cg compiler
pthread_getconcurrency()=0
Version = 3.3.0 NVIDIA 515.105.01
Vendor = NVIDIA Corporation
Renderer = NVIDIA GeForce GTX 1080 Ti/PCIe/SSE2
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
ven = NVIDIA Corporation
ven = NVIDIA Corporation


## Conclusion
By identifying the parameters, WIMUSim can generate high-fidelity virtual IMU data that closely matches real-world recordings.